# STARFIRE Gap Streaming — OpenMC Reproduction
Reproduce Halley & Miller (1986), *Neutron Streaming Through Gaps in Fusion Reactor Shielding*.

**Run order (fresh session):**
1. Cell A — install deps + OpenMC (~15 min first time; ~30 sec from cached wheel); mounts Drive; sets DATA_DIR
2. Cell B — verify import
3. Cell C — nuclear data (one-time ~15 min; skips automatically if already on Drive)
4. Cell D — smoke test (end-to-end check before model run)
5. Sections 1–8 — model, run, results

Geometry: multi-layer slab shield, HFS/MFS/LFS, configurable straight or stepped slot.  
Compare computed dose to Figs. 3–9 of the paper.

---
## Cell A — Install OpenMC
First run: builds from C++ source (~15 min). Subsequent runs: installs from cached wheel on Drive (~30 sec).  
Also mounts Drive and sets DATA_DIR for nuclear data.

In [ ]:
import os, glob
from google.colab import drive

!apt-get install -y libhdf5-dev cmake ninja-build -q

drive.mount("/content/drive")
os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'
DATA_DIR        = "/content/drive/MyDrive/openmc_data/endfb80"
WHEEL_DIR       = "/content/drive/MyDrive/openmc_data/wheel"
SOLIB_CACHE     = "/content/drive/MyDrive/openmc_data/libopenmc.so"
EXEC_CACHE      = "/content/drive/MyDrive/openmc_data/openmc_exec"
OPENMC_LIBDIR   = "/usr/local/lib/python3.12/dist-packages/openmc/lib"
os.makedirs(DATA_DIR,  exist_ok=True)
os.makedirs(WHEEL_DIR, exist_ok=True)

if os.path.exists(SOLIB_CACHE) and os.path.exists(EXEC_CACHE):
    # Fast path (~30 sec)
    wheels = [f for f in os.listdir(WHEEL_DIR) if f.startswith("openmc") and f.endswith(".whl")]
    if wheels:
        !pip install {WHEEL_DIR}/{wheels[0]} -q
    else:
        !pip install openmc==0.15.3 -q
    os.makedirs(OPENMC_LIBDIR, exist_ok=True)
    !cp {SOLIB_CACHE} {OPENMC_LIBDIR}/libopenmc.so
    !cp {SOLIB_CACHE} /usr/local/lib/libopenmc.so && ldconfig
    !cp {EXEC_CACHE} /usr/local/bin/openmc && chmod +x /usr/local/bin/openmc
    print("Fast path done: libopenmc.so + openmc binary restored.")
else:
    # Full CMake build (~20 min, once)
    print("Cloning OpenMC v0.15.3...")
    !git clone --depth 1 --branch v0.15.3 https://github.com/openmc-dev/openmc.git /content/openmc_src -q
    print("CMake configure...")
    !cmake -S /content/openmc_src -B /content/openmc_src/build -DCMAKE_BUILD_TYPE=Release -DOPENMC_USE_OPENMP=OFF 2>&1 | tail -3
    print("Building (~15 min)...")
    !cmake --build /content/openmc_src/build -j2 2>&1 | tail -3
    !pip install /content/openmc_src -q
    # Find and cache libopenmc.so
    sos = [s for s in glob.glob("/content/openmc_src/build/**/*.so*", recursive=True)
           if "openmc" in os.path.basename(s).lower()]
    # Find and cache openmc executable
    exes = [e for e in glob.glob("/content/openmc_src/build/**/openmc", recursive=True)
            if os.path.isfile(e) and os.access(e, os.X_OK) and not e.endswith(".so")]
    print(f"Found .so: {sos}")
    print(f"Found exe: {exes}")
    if sos:
        os.makedirs(OPENMC_LIBDIR, exist_ok=True)
        !cp {sos[0]} {OPENMC_LIBDIR}/libopenmc.so && cp {sos[0]} {SOLIB_CACHE}
        !cp {sos[0]} /usr/local/lib/libopenmc.so && ldconfig
    if exes:
        !cp {exes[0]} /usr/local/bin/openmc && chmod +x /usr/local/bin/openmc
        !cp {exes[0]} {EXEC_CACHE}
    if not sos or not exes:
        print("Some artifacts missing -- listing build output:")
        !find /content/openmc_src/build -maxdepth 3 -name "openmc*" 2>/dev/null

print(f"Data dir: {DATA_DIR}")


---
## Cell B â€” Verify

In [ ]:
import openmc, sys
print(f"OpenMC {openmc.__version__}  |  Python {sys.version.split()[0]}")

---
## Cell C — Nuclear data (one-time, ~15 min)
Downloads ENDF/B-VIII.0 tapes **only for the isotopes we use** and converts them to HDF5.  
Output is ~200–400 MB — fits in 3 GB Drive. Skips automatically on subsequent sessions.

In [ ]:
import os, zipfile, shutil

if "DATA_DIR" not in dir():
    from google.colab import drive; drive.mount("/content/drive")
    DATA_DIR = "/content/drive/MyDrive/openmc_data/endfb80"
WHEEL_DIR  = "/content/drive/MyDrive/openmc_data/wheel"
NJOY_CACHE = "/content/drive/MyDrive/openmc_data/njoy"

try:
    import openmc
except ModuleNotFoundError:
    !apt-get install -y libhdf5-dev -q
    wheels = [f for f in os.listdir(WHEEL_DIR) if f.startswith("openmc") and f.endswith(".whl")]
    !pip install {WHEEL_DIR}/{wheels[0]} -q
    import openmc

XS_FILE = f"{DATA_DIR}/cross_sections.xml"
os.environ["OPENMC_CROSS_SECTIONS"] = XS_FILE

if os.path.exists(XS_FILE) and os.path.getsize(XS_FILE) > 2000:
    print("Nuclear data already present.")
    print(XS_FILE)
else:
    # ── Step 1: NJOY binary ───────────────────────────────────────────
    import shutil as _sh
    if _sh.which("njoy") or _sh.which("njoy21"):
        print("NJOY already in PATH.")
    elif os.path.exists(NJOY_CACHE):
        print("Restoring NJOY from Drive cache...")
        !cp {NJOY_CACHE} /usr/local/bin/njoy && chmod +x /usr/local/bin/njoy
    else:
        print("Building NJOY2016 from source (~5 min)...")
        !apt-get install -y gfortran cmake -q
        !git clone --depth 1 https://github.com/njoy/NJOY2016.git /content/NJOY2016 -q
        !cmake -S /content/NJOY2016 -B /content/NJOY2016/build -DCMAKE_BUILD_TYPE=Release
        !cmake --build /content/NJOY2016/build -j2
        import glob; bins = glob.glob("/content/NJOY2016/build/**/njoy", recursive=True)
        if not bins: raise RuntimeError("njoy binary not found -- check cmake output")
        !cp {bins[0]} /usr/local/bin/njoy
        !cp /usr/local/bin/njoy {NJOY_CACHE}
        print("NJOY built and cached to Drive.")

    # ── Step 2: Download ENDF zip ─────────────────────────────────────
    ENDF_FILES = [
        "n-001_H_001.endf", "n-001_H_002.endf",
        "n-005_B_010.endf", "n-005_B_011.endf",
        "n-006_C_012.endf", "n-006_C_013.endf",
        "n-008_O_016.endf", "n-008_O_017.endf",
        "n-014_Si_028.endf", "n-014_Si_029.endf", "n-014_Si_030.endf",
        "n-022_Ti_046.endf", "n-022_Ti_047.endf", "n-022_Ti_048.endf",
        "n-022_Ti_049.endf", "n-022_Ti_050.endf",
        "n-024_Cr_050.endf", "n-024_Cr_052.endf", "n-024_Cr_053.endf", "n-024_Cr_054.endf",
        "n-025_Mn_055.endf",
        "n-026_Fe_054.endf", "n-026_Fe_056.endf", "n-026_Fe_057.endf", "n-026_Fe_058.endf",
        "n-028_Ni_058.endf", "n-028_Ni_060.endf", "n-028_Ni_061.endf",
        "n-028_Ni_062.endf", "n-028_Ni_064.endf",
        "n-042_Mo_092.endf", "n-042_Mo_094.endf", "n-042_Mo_095.endf",
        "n-042_Mo_096.endf", "n-042_Mo_097.endf", "n-042_Mo_098.endf", "n-042_Mo_100.endf",
    ]
    ZIP_URL    = "https://www.nndc.bnl.gov/endf-b8.0/zips/ENDF-B-VIII.0_neutrons.zip"
    ZIP_LOCAL  = "/content/endfb80_neutrons.zip"
    EXTRACT_DIR = "/content/endfb80_endf"

    if not os.path.exists(ZIP_LOCAL):
        print("Step 2: Downloading ENDF zip (~295 MB)...")
        !wget -q --show-progress -O {ZIP_LOCAL} {ZIP_URL}
    else:
        print("Step 2: Using existing zip.")

    os.makedirs(EXTRACT_DIR, exist_ok=True)
    with zipfile.ZipFile(ZIP_LOCAL) as z:
        all_names = z.namelist()
        for want in ENDF_FILES:
            hits = [n for n in all_names if want in n]
            if hits:
                z.extract(hits[0], EXTRACT_DIR)
            else:
                print(f"  NOT FOUND: {want}")

    # ── Step 3: NJOY processing ───────────────────────────────────────
    print("Step 3: Processing ENDF -> HDF5 via NJOY (~15 min)...")
    os.makedirs(DATA_DIR, exist_ok=True)
    lib = openmc.data.DataLibrary()
    ok, fail = 0, 0
    for root, _, files in os.walk(EXTRACT_DIR):
        for fname in sorted(files):
            if not fname.endswith(".endf"): continue
            path = os.path.join(root, fname)
            try:
                data = openmc.data.IncidentNeutron.from_njoy(path)
                h5 = os.path.join(DATA_DIR, f"{data.name}.h5")
                data.export_to_hdf5(h5, "w")
                lib.register_file(h5)
                print(f"  OK {data.name}")
                ok += 1
            except Exception as e:
                print(f"  FAIL {fname}: {e}")
                fail += 1

    lib.export_to_xml(XS_FILE)
    print(f"Done: {ok} OK, {fail} failed.")
    os.remove(ZIP_LOCAL)
    shutil.rmtree(EXTRACT_DIR, ignore_errors=True)
    print(XS_FILE)


---
## Cell D — Smoke test
Verifies OpenMC + nuclear data end-to-end before building the STARFIRE model.  
Uses only H-1 and O-16 (guaranteed to be in our data set) — no extra downloads.  
**Expected**: non-zero flux printed; assertion passes. If it fails, fix data path before continuing.

In [ ]:
import openmc, os, tempfile
import numpy as np

DATA_DIR = "/content/drive/MyDrive/openmc_data/endfb80"
os.environ["OPENMC_CROSS_SECTIONS"] = f"{DATA_DIR}/cross_sections.xml"

print("openmc binary:")
!which openmc
!openmc --version
print("libopenmc.so:")
!ls -lh /usr/local/lib/python3.12/dist-packages/openmc/lib/
print("xs xml:")
!ls -lh {DATA_DIR}/cross_sections.xml
print("h5 count:")
!ls {DATA_DIR}/*.h5 2>/dev/null | wc -l


with tempfile.TemporaryDirectory() as tmpdir:
    # Material: water
    mat = openmc.Material()
    mat.set_density('g/cm3', 1.0)
    mat.add_nuclide('H1',  2.0, 'ao')
    mat.add_nuclide('O16', 1.0, 'ao')

    # Geometry: 10x10x10 cm box, vacuum boundary
    xm = openmc.XPlane(-5, boundary_type='vacuum')
    xp = openmc.XPlane( 5, boundary_type='vacuum')
    ym = openmc.YPlane(-5, boundary_type='vacuum')
    yp = openmc.YPlane( 5, boundary_type='vacuum')
    zm = openmc.ZPlane(-5, boundary_type='vacuum')
    zp = openmc.ZPlane( 5, boundary_type='vacuum')
    cell = openmc.Cell(fill=mat, region=+xm & -xp & +ym & -yp & +zm & -zp)
    uni  = openmc.Universe(cells=[cell])

    # Source: 14.1 MeV point source at origin
    src = openmc.IndependentSource()
    src.space  = openmc.stats.Point((0, 0, 0))
    src.energy = openmc.stats.Discrete([14.1e6], [1.0])

    s = openmc.Settings()
    s.source    = src
    s.run_mode  = 'fixed source'
    s.batches   = 10
    s.particles = 1000

    tally = openmc.Tally()
    tally.filters = [openmc.CellFilter([cell])]
    tally.scores  = ['flux']

    model = openmc.Model(
        openmc.Geometry(uni),
        openmc.Materials([mat]),
        s,
        openmc.Tallies([tally]),
    )
    sp_path = model.run(cwd=tmpdir, output=True)
    sp = openmc.StatePoint(sp_path)
    flux = sp.get_tally().mean.flat[0]

assert flux > 0 and not np.isnan(flux), f'Bad flux: {flux}'
print(f'OK  flux = {flux:.3e} n/cm\u00b2 per source n')
print(f'OpenMC {openmc.__version__} + nuclear data working correctly.')

In [ ]:
# B10 via ENDF/B-VII.1  (NJOY2016 handles VII.1 reliably; no dlwh law=0 issue)
import openmc.data, os, shutil, zipfile, glob

DATA_DIR   = "/content/drive/MyDrive/openmc_data/endfb80"
NJOY_CACHE = "/content/drive/MyDrive/openmc_data/njoy"

if not shutil.which("njoy"):
    !cp {NJOY_CACHE} /usr/local/bin/njoy && chmod +x /usr/local/bin/njoy
    print("NJOY2016 restored.")

ZIP_URL_A = "https://www.nndc.bnl.gov/endf-b7.1/zips/ENDF-B-VII.1_neutrons.zip"
ZIP_URL_B = "https://www.nndc.bnl.gov/endf-b7.1/zips/ENDF-B-VII.1-neutrons.zip"
ZIP_LOCAL = "/content/endfb71_neutrons.zip"

if not os.path.exists(ZIP_LOCAL) or os.path.getsize(ZIP_LOCAL) < 1_000_000:
    print("Downloading ENDF/B-VII.1 neutrons (~200 MB)...")
    ret = os.system(f"wget -q --show-progress -O {ZIP_LOCAL} {ZIP_URL_A}")
    if ret != 0 or os.path.getsize(ZIP_LOCAL) < 1_000_000:
        print("Trying alternate URL...")
        os.system(f"wget -q --show-progress -O {ZIP_LOCAL} {ZIP_URL_B}")

EXTRACT_DIR = "/content/endfb71_b10"
os.makedirs(EXTRACT_DIR, exist_ok=True)
with zipfile.ZipFile(ZIP_LOCAL) as z:
    hits = [n for n in z.namelist() if "B_010" in n or "B-010" in n]
    print(f"B10 entries in zip: {hits}")
    for h in hits:
        z.extract(h, EXTRACT_DIR)

b10_files = [f for f in glob.glob(f"{EXTRACT_DIR}/**/*", recursive=True)
             if os.path.isfile(f) and ("B_010" in f or "B-010" in f)]
print(f"Extracted: {b10_files}")

if b10_files:
    print("Processing B10 from ENDF/B-VII.1 with NJOY2016 (stdout=True for diagnostics)...")
    try:
        data = openmc.data.IncidentNeutron.from_njoy(b10_files[0], temperatures=[293.6], stdout=True)
        h5 = f"{DATA_DIR}/B10.h5"
        data.export_to_hdf5(h5, "w")
        lib = openmc.data.DataLibrary.from_xml(f"{DATA_DIR}/cross_sections.xml")
        lib.register_file(h5)
        lib.export_to_xml(f"{DATA_DIR}/cross_sections.xml")
        print("B10 (VII.1) OK -- cross_sections.xml updated.")
        print("Now re-run Section 2 (B4C now uses natural boron).")
    except Exception as e:
        print(f"B10 VII.1 NJOY failed: {e}")
else:
    print("No B10 file extracted -- check zip URL or contents.")


---
## 1 â€” Configuration
Change `GAP_TYPE` and `GAP_WIDTH_CM` here; everything below regenerates.

In [ ]:
import os

# â”€â”€ Geometry parameters (from Fig. 2 of the paper) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
HFS_CM   = 50.0   # high-field-side shield thickness
MFS_CM   = 40.0   # mid-field-side shield thickness
LFS_CM   = 18.0   # low-field-side shield thickness
TOTAL_CM = HFS_CM + MFS_CM + LFS_CM   # 108 cm total

SLAB_HALF_WIDTH = 100.0   # half-width in x (200 cm wide slab)

# â”€â”€ Gap configuration â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# GAP_TYPE: 'straight' or 'stepped'
GAP_TYPE      = 'straight'
GAP_WIDTH_CM  = 1.0    # try 1, 2, 5, 10

# For stepped gap: lateral shift per step segment
STEP_OFFSET_CM = 5.0

# â”€â”€ Run settings â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
BATCHES   = 200
PARTICLES = 50_000  # per batch; increase for better statistics

RUN_DIR = f'/content/run_{GAP_TYPE}_{GAP_WIDTH_CM}cm'
os.makedirs(RUN_DIR, exist_ok=True)
print(f"Output: {RUN_DIR}")
print(f"Shield: HFS {HFS_CM} + MFS {MFS_CM} + LFS {LFS_CM} = {TOTAL_CM} cm")
print(f"Gap: {GAP_TYPE}, {GAP_WIDTH_CM} cm wide")

---
## 2 â€” Materials
Verify compositions against Table II of the paper before final runs.

| Material | Density (g/cmÂ³) | Notes |
|---|---|---|
| TiHâ‚‚ | 3.75 | Verify from paper |
| Bâ‚„C | 2.52 | Natural boron |
| Fe-1422 | 8.0 | Treated as 316SS â€” **verify** |
| Hâ‚‚O | 1.0 | Available if paper includes water layer |
| void | ~0 | Gap interior |

In [ ]:
import openmc
import os
os.environ['OPENMC_CROSS_SECTIONS'] = f'{DATA_DIR}/cross_sections.xml'

# â”€â”€ TiHâ‚‚ â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
tih2 = openmc.Material(name='TiH2')
tih2.set_density('g/cm3', 3.75)  # TODO: verify from paper
tih2.add_nuclide('Ti46', 0.0825, 'ao')
tih2.add_nuclide('Ti47', 0.0744, 'ao')
tih2.add_nuclide('Ti48', 0.7372, 'ao')
tih2.add_nuclide('Ti49', 0.0541, 'ao')
tih2.add_nuclide('Ti50', 0.0518, 'ao')
tih2.add_nuclide('H1',   1.9998, 'ao')   # 2 H per Ti; OpenMC normalises ao
tih2.add_nuclide('H2',   0.0002, 'ao')

# â”€â”€ Bâ‚„C â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
b4c = openmc.Material(name='B4C')
b4c.set_density('g/cm3', 2.52)
b4c.add_nuclide('B10',  4 * 0.199, 'ao')  # natural abundance 19.9%
b4c.add_nuclide('B11',  4 * 0.801, 'ao')  # natural abundance 80.1%
b4c.add_nuclide('C12',  0.989, 'ao')
b4c.add_nuclide('C13',  0.011, 'ao')

# â”€â”€ Fe-1422 (placeholder: 316SS â€” verify from paper) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
fe1422 = openmc.Material(name='Fe1422')
fe1422.set_density('g/cm3', 8.0)  # TODO: verify
fe1422.add_element('Fe', 0.655, 'wo')
fe1422.add_element('Cr', 0.170, 'wo')
fe1422.add_element('Ni', 0.120, 'wo')
fe1422.add_element('Mo', 0.025, 'wo')
fe1422.add_element('Mn', 0.020, 'wo')
fe1422.add_element('C',  0.001, 'wo')
fe1422.add_element('Si', 0.009, 'wo')

# â”€â”€ Hâ‚‚O â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
water = openmc.Material(name='H2O')
water.set_density('g/cm3', 1.0)
water.add_nuclide('H1',  2.0,    'ao')
water.add_nuclide('O16', 0.9997, 'ao')
water.add_nuclide('O17', 0.0003, 'ao')

# â”€â”€ Void (gap interior) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
void = openmc.Material(name='void')
void.set_density('g/cm3', 1e-10)
void.add_nuclide('H1', 1.0, 'ao')

materials = openmc.Materials([tih2, b4c, fe1422, water, void])
materials.export_to_xml(path=f'{RUN_DIR}/materials.xml')
print("Materials written.")

---
## 3 â€” Geometry

Coordinate system:
- **z**: beam axis (source â†’ shield â†’ detector)
- **x**: transverse (gap width dimension)
- **y**: height (reflective BC â†’ infinite slab)

Shield layers:
```
z=0        z=50       z=90       z=108
 |-- HFS --|-- MFS --|-- LFS --|
```
Source void: z = -20 to 0  
Detector void: z = 108 to 130  
Gap slot: x = Â±gap_width/2, full z extent of shield

In [ ]:
import openmc

w = GAP_WIDTH_CM / 2.0

# â”€â”€ Axial (z) planes â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
z_src   = openmc.ZPlane(z0=-20.0, boundary_type='vacuum')
z0      = openmc.ZPlane(z0=0.0)
z_hm    = openmc.ZPlane(z0=HFS_CM)
z_ml    = openmc.ZPlane(z0=HFS_CM + MFS_CM)
z1      = openmc.ZPlane(z0=TOTAL_CM)
z_det   = openmc.ZPlane(z0=TOTAL_CM + 22.0, boundary_type='vacuum')

# â”€â”€ Transverse / height planes â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
x_min   = openmc.XPlane(x0=-SLAB_HALF_WIDTH, boundary_type='reflective')
x_max   = openmc.XPlane(x0= SLAB_HALF_WIDTH, boundary_type='reflective')
y_min   = openmc.YPlane(y0=-50.0, boundary_type='reflective')
y_max   = openmc.YPlane(y0= 50.0, boundary_type='reflective')

# â”€â”€ Gap planes â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
x_gm    = openmc.XPlane(x0=-w)
x_gp    = openmc.XPlane(x0= w)

in_box  = +x_min & -x_max & +y_min & -y_max

if GAP_TYPE == 'straight':
    in_gap = +x_gm & -x_gp & +z0 & -z1

elif GAP_TYPE == 'stepped':
    off  = STEP_OFFSET_CM
    xp1m = openmc.XPlane(x0=-w);       xp1p = openmc.XPlane(x0=w)
    xp2m = openmc.XPlane(x0=-w+off);   xp2p = openmc.XPlane(x0=w+off)
    xp3m = openmc.XPlane(x0=-w+2*off); xp3p = openmc.XPlane(x0=w+2*off)
    seg1 = +xp1m & -xp1p & +z0   & -z_hm
    seg2 = +xp2m & -xp2p & +z_hm & -z_ml
    seg3 = +xp3m & -xp3p & +z_ml & -z1
    in_gap = seg1 | seg2 | seg3

else:
    raise ValueError(f"Unknown GAP_TYPE: {GAP_TYPE}")

# â”€â”€ Cells â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
src_cell = openmc.Cell(name='source_void',   fill=None,   region=in_box & +z_src & -z0)
hfs_cell = openmc.Cell(name='HFS',           fill=tih2,   region=in_box & +z0   & -z_hm & ~in_gap)
mfs_cell = openmc.Cell(name='MFS',           fill=b4c,    region=in_box & +z_hm & -z_ml & ~in_gap)
lfs_cell = openmc.Cell(name='LFS',           fill=fe1422, region=in_box & +z_ml & -z1   & ~in_gap)
gap_cell = openmc.Cell(name='gap',           fill=void,   region=in_box & in_gap)
det_cell = openmc.Cell(name='detector_void', fill=None,   region=in_box & +z1   & -z_det)

universe = openmc.Universe(cells=[src_cell, hfs_cell, mfs_cell, lfs_cell, gap_cell, det_cell])
geometry = openmc.Geometry(universe)
geometry.export_to_xml(path=f'{RUN_DIR}/geometry.xml')
print(f"Geometry written.  Gap type={GAP_TYPE}, width={GAP_WIDTH_CM} cm")

---
## 3a â€” Geometry plot
Visual check before the first run. Look for:
- Three distinct shield layers (HFS / MFS / LFS) in different colours
- Narrow gap slot centred at x=0 running the full z extent of the shield
- Source void (z < 0) and detector void (z > 108) at top and bottom

If the gap is invisible or the layers are wrong, fix Section 3 before running.

In [ ]:
import openmc
import matplotlib.pyplot as plt

# Zoom width: show 20Ã— the gap width in x so even a 1 cm slot is visible
plot_width_x = max(GAP_WIDTH_CM * 20, 15.0)
plot_width_z = TOTAL_CM + 44.0   # includes source + detector void

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: zoomed in on gap region
universe.plot(
    basis='xz',
    origin=(0.0, 0.0, TOTAL_CM / 2),
    width=(plot_width_x, plot_width_z),
    pixels=(600, 600),
    color_by='material',
    axes=axes[0],
)
axes[0].set_title(f'Gap region (x \u00b1{plot_width_x/2:.0f} cm)')
axes[0].set_xlabel('x (cm)')
axes[0].set_ylabel('z (cm)')
# Layer boundaries
for z_line in [0, HFS_CM, HFS_CM + MFS_CM, TOTAL_CM]:
    axes[0].axhline(z_line - TOTAL_CM/2, color='white', lw=0.8, ls='--', alpha=0.7)

# Right: full slab width
universe.plot(
    basis='xz',
    origin=(0.0, 0.0, TOTAL_CM / 2),
    width=(SLAB_HALF_WIDTH * 2, plot_width_z),
    pixels=(600, 300),
    color_by='material',
    axes=axes[1],
)
axes[1].set_title('Full slab width (200 cm)')
axes[1].set_xlabel('x (cm)')

plt.suptitle(f'STARFIRE shield â€” {GAP_TYPE} gap, {GAP_WIDTH_CM} cm', fontsize=13)
plt.tight_layout()
plt.savefig(f'{RUN_DIR}/geometry_xz.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {RUN_DIR}/geometry_xz.png")

---
## 4 â€” Source
14.1 MeV mono-energetic neutrons, uniform plane source on the shield front face,
forward-directed (+z). Matches Table I source description.

In [ ]:
import openmc

source = openmc.IndependentSource()
source.space  = openmc.stats.Box(
    lower_left  = [-SLAB_HALF_WIDTH, -50.0, -10.0],
    upper_right = [ SLAB_HALF_WIDTH,  50.0,  -5.0],
)
source.angle  = openmc.stats.Monodirectional(reference_uvw=(0.0, 0.0, 1.0))
source.energy = openmc.stats.Discrete([14.1e6], [1.0])

settings = openmc.Settings()
settings.source    = source
settings.run_mode  = 'fixed source'
settings.batches   = BATCHES
settings.particles = PARTICLES

settings.export_to_xml(path=f'{RUN_DIR}/settings.xml')
print(f"Settings written.  {BATCHES} batches x {PARTICLES:,} particles")

---
## 5 â€” Tallies
Flux tally in the detector void cell, with energy binning.  
Dose is computed by convolving flux with ICRP-116 flux-to-dose coefficients.

In [ ]:
import openmc
import numpy as np

energy_bins = np.logspace(0, np.log10(20e6), 101)

flux_tally = openmc.Tally(name='detector_flux')
flux_tally.filters = [
    openmc.CellFilter([det_cell]),
    openmc.EnergyFilter(energy_bins),
]
flux_tally.scores = ['flux']

total_tally = openmc.Tally(name='detector_total_flux')
total_tally.filters = [openmc.CellFilter([det_cell])]
total_tally.scores  = ['flux']

dose_energies, dose_coeffs = openmc.data.dose_coefficients('neutron', 'AP')
dose_filter = openmc.EnergyFunctionFilter(dose_energies, dose_coeffs)

dose_tally = openmc.Tally(name='detector_dose')
dose_tally.filters = [openmc.CellFilter([det_cell]), dose_filter]
dose_tally.scores  = ['flux']

tallies = openmc.Tallies([flux_tally, total_tally, dose_tally])
tallies.export_to_xml(path=f'{RUN_DIR}/tallies.xml')
print("Tallies written.")

---
## 6 â€” Run

In [ ]:
import openmc, os, shutil

os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'

# Clear stale HDF5 locks from any previous run
shutil.rmtree(RUN_DIR, ignore_errors=True)
os.makedirs(RUN_DIR)

# Re-export XMLs from in-scope objects
materials.export_to_xml(path=f"{RUN_DIR}/materials.xml")
geometry.export_to_xml(path=f"{RUN_DIR}/geometry.xml")
settings.export_to_xml(path=f"{RUN_DIR}/settings.xml")
tallies.export_to_xml(path=f"{RUN_DIR}/tallies.xml")

os.chdir(RUN_DIR)
openmc.run(threads=2, output=True)


---
## 6b — Solid-shield baseline (no gap)
Run once. Provides the denominator for the streaming enhancement ratio.
Expected: very low flux with B10 (may be at statistical noise floor).

In [ ]:
import openmc, os, shutil, glob

os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'
SOLID_DIR = '/content/run_solid'
shutil.rmtree(SOLID_DIR, ignore_errors=True)
os.makedirs(SOLID_DIR)

# Solid-shield geometry -- identical to gap case but no void slot
xmn = openmc.XPlane(x0=-SLAB_HALF_WIDTH, boundary_type='reflective')
xmx = openmc.XPlane(x0= SLAB_HALF_WIDTH, boundary_type='reflective')
ymn = openmc.YPlane(y0=-50.0, boundary_type='reflective')
ymx = openmc.YPlane(y0= 50.0, boundary_type='reflective')
zs  = openmc.ZPlane(z0=-20.0, boundary_type='vacuum')
z0s = openmc.ZPlane(z0=0.0)
zhs = openmc.ZPlane(z0=HFS_CM)
zms = openmc.ZPlane(z0=HFS_CM + MFS_CM)
z1s = openmc.ZPlane(z0=TOTAL_CM)
zds = openmc.ZPlane(z0=TOTAL_CM + 22.0, boundary_type='vacuum')

ib = +xmn & -xmx & +ymn & -ymx
src_s = openmc.Cell(fill=None,   region=ib & +zs  & -z0s)
hfs_s = openmc.Cell(fill=tih2,   region=ib & +z0s & -zhs)
mfs_s = openmc.Cell(fill=b4c,    region=ib & +zhs & -zms)
lfs_s = openmc.Cell(fill=fe1422, region=ib & +zms & -z1s)
det_s = openmc.Cell(fill=None,   region=ib & +z1s & -zds)

openmc.Geometry(openmc.Universe(cells=[src_s, hfs_s, mfs_s, lfs_s, det_s])).export_to_xml(path=f'{SOLID_DIR}/geometry.xml')
materials.export_to_xml(path=f'{SOLID_DIR}/materials.xml')
settings.export_to_xml(path=f'{SOLID_DIR}/settings.xml')

ft = openmc.Tally(name='detector_total_flux')
ft.filters = [openmc.CellFilter([det_s])]
ft.scores  = ['flux']
openmc.Tallies([ft]).export_to_xml(path=f'{SOLID_DIR}/tallies.xml')

os.chdir(SOLID_DIR)
openmc.run(threads=2, output=True)
print("Solid-shield run complete.")


---
## 7 â€” Results

In [ ]:
import openmc, glob
import numpy as np

DET_VOLUME = 200.0 * 100.0 * 22.0  # cm^3  (x: 200, y: 100, z: 22)

# --- Gap case ---
sp_gap = openmc.StatePoint(sorted(glob.glob(f'{RUN_DIR}/statepoint.*.h5'))[-1])
tg = sp_gap.get_tally(name='detector_total_flux')
gap_vi  = tg.mean.flat[0]               # n*cm per source n (volume-integrated)
gap_err = tg.std_dev.flat[0]
gap_avg = gap_vi / DET_VOLUME            # n/cm^2 per source n (average flux)

# --- Solid-shield case ---
solid_files = sorted(glob.glob('/content/run_solid/statepoint.*.h5'))
if solid_files:
    sp_solid = openmc.StatePoint(solid_files[-1])
    ts = sp_solid.get_tally(name='detector_total_flux')
    sol_vi  = ts.mean.flat[0]
    sol_err = ts.std_dev.flat[0]
    sol_avg = sol_vi / DET_VOLUME
    ratio   = gap_avg / sol_avg if sol_avg > 0 else float('inf')
else:
    sol_vi = sol_avg = sol_err = 0.0
    ratio  = float('inf')
    print("No solid-shield statepoint found -- run Section 6b first.")

# --- Dose (gap case, unnormalized) ---
td = sp_gap.get_tally(name='detector_dose')
dose_vi = td.mean.flat[0]

print(f"Detector volume: {DET_VOLUME:.0f} cm^3\n")
print(f"{'Case':<28} {'Vol-int (n*cm)':>16} {'Avg flux (n/cm2)':>18} {'Rel err':>8}")
print("-" * 74)
print(f"{'Gap ' + GAP_TYPE + ' ' + str(GAP_WIDTH_CM) + ' cm':<28} {gap_vi:>16.4e} {gap_avg:>18.4e} {gap_err/gap_vi:>7.1%}")
if solid_files:
    rel = sol_err/sol_vi if sol_vi > 0 else float('nan')
    print(f"{'Solid shield (no gap)':<28} {sol_vi:>16.4e} {sol_avg:>18.4e} {rel:>7.1%}")
    print(f"\nStreaming enhancement (gap/solid):   {ratio:.2e}")
    print(f"Paper (1 cm straight slot):          ~1e+10  (Figs 3-9)")

print(f"\nDose (unnorm):  {dose_vi:.4e} pSv*cm^2 per source n")
print("\nNote: solid-shield flux near noise floor with B10 -- ratio is a lower bound.")
print("Variance reduction (CADIS/weight windows) needed for solid-shield accuracy.")


---
## 8 â€” Batch runner (all gap widths)
Sweeps all four straight-slot widths sequentially.

In [ ]:
import openmc, os, shutil, glob
import numpy as np

os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'
results = {}

for gap_w in [1.0, 2.0, 5.0, 10.0]:
    run_dir = f'/content/run_straight_{gap_w}cm'
    # Clear stale HDF5 files from any previous run
    shutil.rmtree(run_dir, ignore_errors=True)
    os.makedirs(run_dir)

    ww = gap_w / 2.0
    xgm = openmc.XPlane(x0=-ww);  xgp = openmc.XPlane(x0=ww)
    ig  = +xgm & -xgp & +z0 & -z1
    ib  = +x_min & -x_max & +y_min & -y_max

    src_c = openmc.Cell(fill=None,   region=ib & +z_src & -z0)
    hfs_c = openmc.Cell(fill=tih2,   region=ib & +z0    & -z_hm & ~ig)
    mfs_c = openmc.Cell(fill=b4c,    region=ib & +z_hm  & -z_ml & ~ig)
    lfs_c = openmc.Cell(fill=fe1422, region=ib & +z_ml  & -z1   & ~ig)
    gap_c = openmc.Cell(fill=void,   region=ib & ig)
    det_c = openmc.Cell(fill=None,   region=ib & +z1    & -z_det)

    openmc.Geometry(openmc.Universe(cells=[src_c, hfs_c, mfs_c, lfs_c, gap_c, det_c])).export_to_xml(path=f'{run_dir}/geometry.xml')
    materials.export_to_xml(path=f'{run_dir}/materials.xml')
    settings.export_to_xml(path=f'{run_dir}/settings.xml')

    ft = openmc.Tally(name='detector_total_flux')
    ft.filters = [openmc.CellFilter([det_c])]
    ft.scores  = ['flux']
    openmc.Tallies([ft]).export_to_xml(path=f'{run_dir}/tallies.xml')

    os.chdir(run_dir)
    openmc.run(threads=2, output=False)

    sp_file = sorted(glob.glob(f'{run_dir}/statepoint.*.h5'))[-1]
    sp = openmc.StatePoint(sp_file)
    t  = sp.get_tally(name='detector_total_flux')
    results[gap_w] = (t.mean.flat[0], t.std_dev.flat[0])
    print(f"gap={gap_w:4.1f} cm  flux={t.mean.flat[0]:.4e} +/- {t.std_dev.flat[0]:.2e}  n*cm/source-n")

DET_VOLUME = 200.0 * 100.0 * 22.0
print(f"\n{'Gap (cm)':>10} {'Avg flux (n/cm2)':>20} {'Rel err':>10}")
print("-" * 44)
for gap_w, (mean, std) in sorted(results.items()):
    print(f"{gap_w:>10.1f} {mean/DET_VOLUME:>20.4e} {std/mean:>9.1%}")


---
## 8b — Batch runner: stepped configurations
Sweeps gap widths [1, 2, 5] cm x step offsets [2.5, 5, 10] cm = 9 runs.
Expected runtime: ~90 min total.

In [ ]:
import openmc, os, shutil, glob, json as _json
import numpy as np

os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'
DET_VOLUME   = 200.0 * 100.0 * 22.0
RESULTS_FILE = '/content/drive/MyDrive/openmc_data/stepped_results.json'
WIDTHS       = [1.0, 2.0, 5.0]
OFFSETS      = [2.5, 5.0, 10.0]

# Load any results already saved to Drive
if os.path.exists(RESULTS_FILE):
    with open(RESULTS_FILE) as f:
        saved = _json.load(f)
    results_stepped = {tuple(map(float, k.split(','))): v for k, v in saved.items()}
    print(f"Loaded {len(results_stepped)} saved results from Drive.")
else:
    results_stepped = {}

for gap_w in WIDTHS:
    for step_off in OFFSETS:
        key = (gap_w, step_off)
        if key in results_stepped:
            mean, std = results_stepped[key]
            print(f"SKIP  w={gap_w:.1f}cm off={step_off:.1f}cm  (already on Drive: {mean:.4e})")
            continue

        run_dir = f'/content/run_stepped_{gap_w}cm_{step_off}off'
        shutil.rmtree(run_dir, ignore_errors=True)
        os.makedirs(run_dir)

        w    = gap_w / 2.0
        xp1m = openmc.XPlane(x0=-w);            xp1p = openmc.XPlane(x0=w)
        xp2m = openmc.XPlane(x0=-w+step_off);   xp2p = openmc.XPlane(x0=w+step_off)
        xp3m = openmc.XPlane(x0=-w+2*step_off); xp3p = openmc.XPlane(x0=w+2*step_off)
        seg1   = +xp1m & -xp1p & +z0   & -z_hm
        seg2   = +xp2m & -xp2p & +z_hm & -z_ml
        seg3   = +xp3m & -xp3p & +z_ml & -z1
        in_gap = seg1 | seg2 | seg3
        ib     = +x_min & -x_max & +y_min & -y_max

        src_c = openmc.Cell(fill=None,   region=ib & +z_src & -z0)
        hfs_c = openmc.Cell(fill=tih2,   region=ib & +z0    & -z_hm & ~in_gap)
        mfs_c = openmc.Cell(fill=b4c,    region=ib & +z_hm  & -z_ml & ~in_gap)
        lfs_c = openmc.Cell(fill=fe1422, region=ib & +z_ml  & -z1   & ~in_gap)
        gap_c = openmc.Cell(fill=void,   region=ib & in_gap)
        det_c = openmc.Cell(fill=None,   region=ib & +z1    & -z_det)

        openmc.Geometry(openmc.Universe(cells=[src_c, hfs_c, mfs_c, lfs_c, gap_c, det_c])).export_to_xml(path=f'{run_dir}/geometry.xml')
        materials.export_to_xml(path=f'{run_dir}/materials.xml')
        settings.export_to_xml(path=f'{run_dir}/settings.xml')
        ft = openmc.Tally(name='detector_total_flux')
        ft.filters = [openmc.CellFilter([det_c])]
        ft.scores  = ['flux']
        openmc.Tallies([ft]).export_to_xml(path=f'{run_dir}/tallies.xml')

        os.chdir(run_dir)
        openmc.run(threads=2, output=False)

        sp_file = sorted(glob.glob(f'{run_dir}/statepoint.*.h5'))[-1]
        sp = openmc.StatePoint(sp_file)
        t  = sp.get_tally(name='detector_total_flux')
        mean, std = float(t.mean.flat[0]), float(t.std_dev.flat[0])
        results_stepped[key] = [mean, std]

        # Save to Drive after every run
        with open(RESULTS_FILE, 'w') as f:
            _json.dump({f'{k[0]},{k[1]}': v for k, v in results_stepped.items()}, f, indent=2)

        print(f"w={gap_w:.1f}cm off={step_off:.1f}cm  flux={mean:.4e} +/- {std:.2e}  ({std/mean:.1%})  [saved]")

print(f"\n{'Gap (cm)':>9} {'Offset (cm)':>12} {'Vol-int (n*cm)':>16} {'Avg flux (n/cm2)':>18} {'Rel err':>8}")
print("-" * 68)
for (gap_w, step_off), (mean, std) in sorted(results_stepped.items()):
    print(f"{gap_w:>9.1f} {step_off:>12.1f} {mean:>16.4e} {mean/DET_VOLUME:>18.4e} {std/mean:>7.1%}")


---
## 9 — ML Surrogate (Gaussian Process)
Train a GP on the 13-point parametric dataset (4 straight + 9 stepped).  
Features: `gap_type`, `gap_width`, `step_offset`, `overlap_ratio` (derived).  
Target: `log₁₀(flux)` — working in log-space keeps the GP well-conditioned across 4 decades.

**Leave-one-out cross-validation** (13 folds) gives an honest estimate of predictive error with this small dataset.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel
from sklearn.preprocessing import StandardScaler

# ── Complete dataset ──────────────────────────────────────────────────────────
raw = [
    (0,  1.0,  0.0, 1.1123e-1),
    (0,  2.0,  0.0, 2.2145e-1),
    (0,  5.0,  0.0, 5.5724e-1),
    (0, 10.0,  0.0, 1.1243e+0),
    (1,  1.0,  2.5, 3.6202e-3),
    (1,  1.0,  5.0, 3.5729e-3),
    (1,  1.0, 10.0, 3.2396e-3),
    (1,  2.0,  2.5, 9.2993e-3),
    (1,  2.0,  5.0, 7.3288e-3),
    (1,  2.0, 10.0, 6.3946e-3),
    (1,  5.0,  2.5, 2.9945e-1),   # overlap regime
    (1,  5.0,  5.0, 4.1717e-2),
    (1,  5.0, 10.0, 2.0238e-2),
]

data = np.array([[r[0], r[1], r[2]] for r in raw], dtype=float)
flux = np.array([r[3] for r in raw])

# ── Feature engineering ───────────────────────────────────────────────────────
# log10(width): straight flux ∝ width exactly, so log(flux) is linear in
# log(width). The GP interpolates/extrapolates correctly in this coordinate.
log_width = np.log10(data[:, 1])

# overlap_ratio: fraction of gap with unobstructed line of sight
overlap = np.where(
    data[:, 0] == 0,
    1.0,
    np.maximum(0.0, 1.0 - 2.0 * data[:, 2] / data[:, 1])
)

X = np.column_stack([data[:, 0], log_width, data[:, 2], overlap])
y = np.log10(flux)

n      = len(y)
labels = [
    f"str {r[1]:.0f}cm" if r[0] == 0 else f"step {r[1]:.0f}cm/{r[2]:.0f}off"
    for r in raw
]
colors = ['#3b82f6' if r[0] == 0 else '#f59e0b' for r in raw]

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ── GP — alpha jitter replaces WhiteKernel, no ConvergenceWarning ─────────────
kernel = (
    ConstantKernel(1.0, (1e-3, 1e3))
    * RBF(length_scale=np.ones(X.shape[1]),
          length_scale_bounds=[(1e-2, 1e2)] * X.shape[1])
)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-8,
                               n_restarts_optimizer=10,
                               normalize_y=True, random_state=42)
gp.fit(X_scaled, y)
print(f"Full-data GP fitted.\nKernel: {gp.kernel_}\n")

# ── Leave-one-out cross-validation ────────────────────────────────────────────
y_pred_loo = np.zeros(n)
y_std_loo  = np.zeros(n)

for i in range(n):
    mask = np.ones(n, dtype=bool); mask[i] = False
    gp_i = GaussianProcessRegressor(kernel=kernel, alpha=1e-8,
                                     n_restarts_optimizer=5,
                                     normalize_y=True, random_state=42)
    gp_i.fit(X_scaled[mask], y[mask])
    mu, sig = gp_i.predict(X_scaled[[i]], return_std=True)
    y_pred_loo[i] = mu[0]
    y_std_loo[i]  = sig[0]

flux_true = 10 ** y
flux_pred = 10 ** y_pred_loo
flux_lo   = 10 ** (y_pred_loo - 2 * y_std_loo)
flux_hi   = 10 ** (y_pred_loo + 2 * y_std_loo)
err_dec   = y_pred_loo - y

# ── Plots ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for ft, fp, fl, fh, lbl, c in zip(flux_true, flux_pred, flux_lo, flux_hi, labels, colors):
    ax.errorbar(ft, fp, yerr=[[fp - fl], [fh - fp]],
                fmt='o', color=c, capsize=4, markersize=7, label=lbl)
lo = min(flux_true.min(), flux_pred.min()) * 0.3
hi = max(flux_true.max(), flux_pred.max()) * 3.0
ax.plot([lo, hi], [lo, hi], 'k--', lw=1, label='perfect')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlim(lo, hi);  ax.set_ylim(lo, hi)
ax.set_xlabel('OpenMC flux  (n·cm / source·n)')
ax.set_ylabel('GP prediction  (LOO)')
ax.set_title('LOO cross-validation — parity plot')
ax.legend(fontsize=7, ncol=2)
ax.text(0.03, 0.97, 'Error bars: ±2σ GP uncertainty', transform=ax.transAxes,
        fontsize=7, va='top', color='gray')

ax2 = axes[1]
ax2.barh(range(n), err_dec, color=colors)
ax2.axvline(0,    color='k',  lw=1)
ax2.axvline(-0.5, color='r',  lw=0.8, ls='--', label='±0.5 decade')
ax2.axvline( 0.5, color='r',  lw=0.8, ls='--')
ax2.set_yticks(range(n))
ax2.set_yticklabels(labels, fontsize=8)
ax2.set_xlabel('log₁₀(predicted / true)  [decades]')
ax2.set_title('LOO residuals  (0 = perfect)')
ax2.legend(fontsize=8)

plt.suptitle('STARFIRE GP Surrogate — LOO CV  (v3: log₁₀(width), alpha jitter)', fontsize=13)
plt.tight_layout()
plt.savefig('/content/gp_loo.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Summary ───────────────────────────────────────────────────────────────────
abs_err = np.abs(err_dec)
print(f"LOO summary ({n} points, features: gap_type, log10_width, offset, overlap):")
print(f"  Mean |error|:      {abs_err.mean():.2f} decades")
print(f"  Max  |error|:      {abs_err.max():.2f} decades  ({labels[np.argmax(abs_err)]})")
print(f"  Within ±0.5 dec:   {(abs_err <= 0.5).sum()}/{n}")
print(f"  Within ±1.0 dec:   {(abs_err <= 1.0).sum()}/{n}")
print()
for lbl, ed in zip(labels, err_dec):
    bar = '█' * int(abs(ed) * 10)
    sign = '+' if ed >= 0 else '-'
    print(f"  {lbl:<22} {sign}{abs(ed):.2f} dec  {bar}")

# ── Predict new configurations ────────────────────────────────────────────────
def predict(gap_type, width, offset):
    ov = 1.0 if gap_type == 0 else max(0.0, 1.0 - 2 * offset / width)
    x  = scaler.transform([[gap_type, np.log10(width), offset, ov]])
    mu, sig = gp.predict(x, return_std=True)
    return 10**mu[0], 10**(mu[0] - 2*sig[0]), 10**(mu[0] + 2*sig[0])

print("\nExample predictions (full GP, not LOO):")
cases = [
    ("straight 3 cm",    0, 3.0, 0.0),
    ("straight 7 cm",    0, 7.0, 0.0),
    ("stepped 3cm/7off", 1, 3.0, 7.0),
    ("stepped 2cm/3off", 1, 2.0, 3.0),
    ("stepped 5cm/7off", 1, 5.0, 7.0),
]
for name, gt, w, off in cases:
    fp, fl, fh = predict(gt, w, off)
    print(f"  {name:<22}  flux = {fp:.3e}  [{fl:.2e}, {fh:.2e}]  (±2σ)")


---
## Notes / TODO

1. **Fe-1422 material** â€” confirm exact composition and density from Table II of the paper.  
   Current placeholder is 316SS; update `fe1422` cell if STARFIRE used a different alloy.

2. **Source spectrum** â€” Table I may show a multi-group spectrum, not pure 14.1 MeV.  
   If so, replace `openmc.stats.Discrete` with `openmc.stats.Tabular(E_bins, probs)`.

3. **Dose normalization** â€” `detector_dose` tally needs normalization to rem/h per source n/cmÂ²/s  
   to match paper figures. Multiply by source intensity; divide by detector cell volume.

4. **Variance reduction** â€” 108 cm deep penetration needs weight windows or CADIS.  
   Activate `openmc.WeightWindows` if detector statistics are poor (rel. error > 20%).

5. **Hâ‚‚O layer** â€” check Fig. 2; if the paper includes a water layer, add a `water` cell  
   between two of the existing shield layers.

6. **Geometry plot axes** â€” `universe.plot()` with `axes=` requires OpenMC >= 0.13.3.  
   If it raises TypeError, remove the `axes=` arguments and call plt.subplot() separately.

7. **Phase 5** â€” once cases match, replace manual loop with a parametric sweep  
   (gap width Ã— offset) to generate ML training data.